# Notebook 04 — GRU Training
Trains the GRU baseline using the same hyperparameter grid as LSTM for fair comparison.

In [ ]:
import subprocess, os
from pathlib import Path

REPO_URL  = 'https://github.com/calvinkatoroy/tugas-akhir-ai-kel-08.git'
REPO_NAME = 'tugas-akhir-ai-kel-08'

cwd = Path.cwd()

if (cwd / '../src').resolve().exists():
    result = subprocess.run(['git', '-C', str((cwd / '..').resolve()), 'pull'],
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

elif (cwd / 'src').exists():
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
    os.chdir('notebooks')

else:
    repo_path = cwd / REPO_NAME
    if not repo_path.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    else:
        print('Repo found, pulling latest...')
        subprocess.run(['git', '-C', str(repo_path), 'pull'], capture_output=True)
    os.chdir(repo_path / 'notebooks')

print(f'Working dir: {Path.cwd()}')

In [ ]:
import sys
sys.path.insert(0, '..')

import random
import copy
import numpy as np
import torch
import yaml
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
def safe_log_experiment(record, csv_path):
    import os, pandas as pd
    from pathlib import Path
    path = Path(csv_path)
    if path.exists():
        try:
            pd.read_csv(path)
        except Exception:
            print(f'WARNING: corrupt CSV at {path}, resetting.')
            os.remove(path)
    from src.utils import log_experiment
    log_experiment(record, csv_path=csv_path)


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/tugas-akhir-ai")
    SPLITS_DIR = DRIVE_ROOT / "splits"
else:
    import yaml
    with open("../config.yaml") as f:
        _cfg = yaml.safe_load(f)
    DRIVE_ROOT = Path(_cfg["data"]["drive_root"])
    SPLITS_DIR = Path(_cfg["data"]["splits_drive"])

print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"SPLITS_DIR: {SPLITS_DIR}")

In [ ]:
with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

X_train = np.load(SPLITS_DIR / "X_train_seq.npy")
y_train = np.load(SPLITS_DIR / "y_train_seq.npy")
X_val   = np.load(SPLITS_DIR / "X_val_seq.npy")
y_val   = np.load(SPLITS_DIR / "y_val_seq.npy")
X_test  = np.load(SPLITS_DIR / "X_test_seq.npy")
y_test  = np.load(SPLITS_DIR / "y_test_seq.npy")

n_features = X_train.shape[2]
seq_len    = X_train.shape[1]
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"n_features={n_features}, seq_len={seq_len}")

## Baseline run (config defaults)

In [ ]:
from src.models.gru_model import build_gru
from src.train import train_model
from src.utils import log_experiment

model = build_gru(cfg, n_features)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

In [ ]:
import os
_ckpt = '../results/checkpoints/best_gru.pt'
if os.path.exists(_ckpt):
    print(f'Checkpoint {_ckpt} already exists — skipping baseline training.')
    history_baseline = {'val_f1': [0.0], 'train_loss': [], 'val_loss': []}
else:
    history_baseline, ckpt_baseline = train_model(
        model, X_train, y_train, X_val, y_val,
        cfg=cfg, model_key='gru',
        checkpoint_dir='../results/checkpoints',
    )

    safe_log_experiment({
        'exp_id': 'gru_01_baseline',
        'model': 'GRU',
        'hidden_size': cfg['gru']['hidden_size'],
        'num_layers': cfg['gru']['num_layers'],
        'dropout': cfg['gru']['dropout'],
        'seq_len': cfg['gru']['seq_len'],
        'lr': cfg['gru']['learning_rate'],
        'batch_size': cfg['gru']['batch_size'],
        'best_val_f1': round(max(history_baseline['val_f1']), 4),
        'notes': 'baseline — config defaults',
    }, csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))

## Hyperparameter Experiments

In [ ]:
def run_gru_experiment(exp_id, overrides, notes=''):
    exp_cfg = copy.deepcopy(cfg)
    exp_cfg['gru'].update(overrides)

    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    if torch.cuda.is_available(): torch.cuda.empty_cache()
    m = build_gru(exp_cfg, n_features)
    hist, ckpt = train_model(
        m, X_train, y_train, X_val, y_val,
        cfg=exp_cfg, model_key='gru',
        checkpoint_dir='../results/checkpoints',
    )
    best_f1 = round(max(hist['val_f1']), 4)
    safe_log_experiment({
        'exp_id': exp_id,
        'model': 'GRU',
        'hidden_size': exp_cfg['gru']['hidden_size'],
        'num_layers': exp_cfg['gru']['num_layers'],
        'dropout': exp_cfg['gru']['dropout'],
        'seq_len': exp_cfg['gru']['seq_len'],
        'lr': exp_cfg['gru']['learning_rate'],
        'batch_size': exp_cfg['gru']['batch_size'],
        'best_val_f1': best_f1,
        'notes': notes,
    }, csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))
    print(f'[{exp_id}] best_val_f1={best_f1}')
    return hist, ckpt, best_f1

In [ ]:
hist_02, _, f1_02 = run_gru_experiment('gru_02_hidden64',  {'hidden_size': 64},  'hidden_size=64')

In [ ]:
hist_03, _, f1_03 = run_gru_experiment('gru_03_hidden256', {'hidden_size': 256}, 'hidden_size=256')

In [ ]:
hist_04, _, f1_04 = run_gru_experiment('gru_04_layers3',   {'num_layers': 3},    'num_layers=3')

In [ ]:
hist_05, _, f1_05 = run_gru_experiment('gru_05_lr1e4',     {'learning_rate': 1e-4}, 'lr=1e-4')

In [ ]:
# Exp 06 — dropout=0.5
hist_06, _, f1_06 = run_gru_experiment('gru_06_dropout05', {'dropout': 0.5}, 'dropout=0.5')

## Final Evaluation on Test Set

In [ ]:
import pandas as pd

results = pd.read_csv(str(DRIVE_ROOT / "metrics_summary.csv"))
gru_results = results[results["model"] == "GRU"].copy()
print(gru_results[["exp_id", "hidden_size", "num_layers", "dropout",
                    "seq_len", "lr", "best_val_f1", "notes"]])

best_row = gru_results.loc[gru_results["best_val_f1"].idxmax()]
print(f'
Best experiment: {best_row["exp_id"]}  val_F1={best_row["best_val_f1"]}')

_hist_map = {
    "gru_01_baseline": history_baseline,
    "gru_02_hidden64": hist_02,
    "gru_03_hidden256": hist_03,
    "gru_04_layers3": hist_04,
    "gru_05_lr1e4": hist_05,
    "gru_06_dropout05": hist_06,
}
best_history = _hist_map.get(best_row["exp_id"], history_baseline)

In [ ]:
from src.models.gru_model import build_gru
from src.evaluate import evaluate_dl_model
from src.utils import plot_loss_curves

best_model = build_gru(cfg, n_features).to(device)
best_model.load_state_dict(torch.load('../results/checkpoints/best_gru.pt', map_location=device))

y_pred_gru, y_prob_gru = evaluate_dl_model(
    best_model, X_test, y_test,
    model_name='GRU',
    history=history_baseline,
    figures_dir='../results/figures',
    csv_path=str(DRIVE_ROOT / "metrics_summary.csv"),
)

np.save('../results/gru_y_prob.npy', y_prob_gru)

plot_loss_curves(
    best_history['train_loss'], best_history['val_loss'],
    title='GRU Loss Curves (baseline)',
    save_path='../results/figures/loss_gru.png',
)

In [ ]:
# Save best checkpoint to Drive so it persists across sessions
import shutil
shutil.copy("../results/checkpoints/best_gru.pt", str(DRIVE_ROOT / "best_gru.pt"))
print(f"Checkpoint saved to Drive: {DRIVE_ROOT / "best_gru.pt"}")